Quy trình Prepocessing:
1.Đọc dữ liệu thô
2.Chuẩn hóa cấu trúc (Tách category, company, location)
3.Chuẩn hóa kiểu dữ liệu(Numeric, text, datetime)
4.Xử lí duplicatied 
5.Kiểm tra tính hợp lệ(Salary, đơn vị lương và dữ liệu địa lí)
6.Xử lí missing value
7.Loại metada ko cần thiết 
8.Lưu dữ liệu đã được xử lí sang csv 

Bước 1: đọc dữ liệu thô

In [1]:
import pandas as pd
from pathlib import Path
import ast

# 1. Xác định thư mục data
ROOT = Path.cwd()
DATA_DIR = ROOT/ ".." / "data" / "raw"

# 2. Đọc dữ liệu csv
CSV_PATH = DATA_DIR / "raw_adzuna_data.csv"

# 3. Đọc dữ liệu
df = pd.read_csv(
    CSV_PATH,
    dtype={"id": "string"}
)

# 4. Kiểm tra dữ liệu sau khi đọc
print("Dataset Shape:", df.shape)
display(df.head())


Dataset Shape: (2403, 17)


,id,contract_type,redirect_url,salary_max,contract_time,latitude,created,category,longitude,title,location,description,salary_min,adref,salary_is_predicted,company,__CLASS__
0,5904487469,permanent,https://www.adzuna.com/land/ad/5904487469?se=3...,312845.06,full_time,37.393080,2026-09-30T22:39:48Z,"{'tag': 'it-jobs', 'label': 'IT Jobs', '__CLAS...",-122.077020,Principal Software Engineer,"{'area': ['US', 'California', 'Santa Clara Cou...",Microsoft and OpenAI have formed one of the mo...,312845.06,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiM09WQkNkdko4UkdIV...,1,"{'display_name': 'Microsoft', '__CLASS__': 'Ad...",Adzuna::API::Response::Job
1,5903572359,permanent,https://www.adzuna.com/details/5903572359?utm_...,99000.00,full_time,33.751486,2026-09-30T02:32:52Z,"{'tag': 'engineering-jobs', 'label': 'Engineer...",-117.754930,Software Engineer,{'__CLASS__': 'Adzuna::API::Response::Location...,ETAP empowers customers to make informed decis...,59500.00,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiM09WQkNkdko4UkdIV...,0,{'__CLASS__': 'Adzuna::API::Response::Company'...,Adzuna::API::Response::Job
2,5879639363,permanent,https://www.adzuna.com/details/5879639363?utm_...,106080.02,full_time,37.798085,2026-09-11T07:15:48Z,"{'label': 'IT Jobs', 'tag': 'it-jobs', '__CLAS...",-122.466538,Software Engineer,{'__CLASS__': 'Adzuna::API::Response::Location...,"Responsibilities Design, implement, and mainta...",106080.02,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiM09WQkNkdko4UkdIV...,1,{'__CLASS__': 'Adzuna::API::Response::Company'...,Adzuna::API::Response::Job
3,5887617106,permanent,https://www.adzuna.com/details/5887617106?utm_...,227000.00,full_time,38.664532,2026-09-17T07:29:42Z,"{'label': 'IT Jobs', 'tag': 'it-jobs', '__CLAS...",-121.145000,Software Engineer,{'__CLASS__': 'Adzuna::API::Response::Location...,"Voyager is an innovative space, defense, and n...",104000.00,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTg4NzYxNzEwNiIsI...,0,{'__CLASS__': 'Adzuna::API::Response::Company'...,Adzuna::API::Response::Job
4,5885956386,permanent,https://www.adzuna.com/details/5885956386?utm_...,121100.00,full_time,34.508431,2026-09-16T06:30:22Z,"{'label': 'Engineering Jobs', 'tag': 'engineer...",-119.731021,Software Engineer,"{'display_name': 'Goleta, Santa Barbara County...",Be visionary Teledyne Technologies Incorporate...,90800.00,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiM09WQkNkdko4UkdIV...,0,"{'display_name': 'Teledyne Technologies', '__C...",Adzuna::API::Response::Job


Bước 2: Flatten Nested Columns (Chuẩn hóa cấu trúc dữ liệu)

Trong dữ liệu thô Adzuna, các cột category, company và location chứa thông tin dưới dạng chuỗi biểu diễn dictionary, thay vì các giá trị đơn lẻ.

Bước này nhằm chuyển đổi các cột có cấu trúc lồng nhau thành những cột riêng biệt, giúp dữ liệu dễ truy xuất, phân tích và sử dụng trong các bước tiếp theo.

Các thao tác thực hiện:

Sử dụng ast.literal_eval() để chuyển chuỗi thành dictionary Python.

Trích xuất thông tin từ category thành category_label và category_tag.

Trích xuất tên công ty từ company thành company_name.

Trích xuất thông tin địa lý từ location thành location_name, location_country và location_state.

Giữ nguyên các cột gốc và cột trung gian để đối chiếu, kiểm tra trước khi loại bỏ ở Cell 7.

Kết quả mong đợi: Các thông tin lồng nhau được tách thành những cột có cấu trúc rõ ràng, thuận tiện cho việc làm sạch và phân tích dữ liệu tiếp theo.

In [2]:
# Chuyển chuỗi dictionary thành dictionary Python
df["category_obj"] = df["category"].apply(ast.literal_eval)
df["company_obj"] = df["company"].apply(ast.literal_eval)
df["location_obj"] = df["location"].apply(ast.literal_eval)

# Category
df["category_label"] = df["category_obj"].apply(
    lambda x: x.get("label")
)

df["category_tag"] = df["category_obj"].apply(
    lambda x: x.get("tag")
)

# Company
df["company_name"] = df["company_obj"].apply(
    lambda x: x.get("display_name")
)

# Location
df["location_name"] = df["location_obj"].apply(
    lambda x: x.get("display_name")
)

df["location_country"] = df["location_obj"].apply(
    lambda x: x["area"][0]
    if len(x.get("area") or []) >= 1 else None
)

df["location_state"] = df["location_obj"].apply(
    lambda x: x["area"][1]
    if len(x.get("area") or []) >= 2 else None
)

# Xóa các cột trung gian
df = df.drop(columns=[
    "category_obj",
    "company_obj",
    "location_obj"
])

display(df.head())

,id,contract_type,redirect_url,salary_max,contract_time,latitude,created,category,longitude,title,...,adref,salary_is_predicted,company,__CLASS__,category_label,category_tag,company_name,location_name,location_country,location_state
0,5904487469,permanent,https://www.adzuna.com/land/ad/5904487469?se=3...,312845.06,full_time,37.393080,2026-09-30T22:39:48Z,"{'tag': 'it-jobs', 'label': 'IT Jobs', '__CLAS...",-122.077020,Principal Software Engineer,...,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiM09WQkNkdko4UkdIV...,1,"{'display_name': 'Microsoft', '__CLASS__': 'Ad...",Adzuna::API::Response::Job,IT Jobs,it-jobs,Microsoft,"Mountain View, Santa Clara County",US,California
1,5903572359,permanent,https://www.adzuna.com/details/5903572359?utm_...,99000.00,full_time,33.751486,2026-09-30T02:32:52Z,"{'tag': 'engineering-jobs', 'label': 'Engineer...",-117.754930,Software Engineer,...,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiM09WQkNkdko4UkdIV...,0,{'__CLASS__': 'Adzuna::API::Response::Company'...,Adzuna::API::Response::Job,Engineering Jobs,engineering-jobs,ETAP,"Irvine, Orange County",US,California
2,5879639363,permanent,https://www.adzuna.com/details/5879639363?utm_...,106080.02,full_time,37.798085,2026-09-11T07:15:48Z,"{'label': 'IT Jobs', 'tag': 'it-jobs', '__CLAS...",-122.466538,Software Engineer,...,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiM09WQkNkdko4UkdIV...,1,{'__CLASS__': 'Adzuna::API::Response::Company'...,Adzuna::API::Response::Job,IT Jobs,it-jobs,Pivot Robotics,"San Francisco, California",US,California
3,5887617106,permanent,https://www.adzuna.com/details/5887617106?utm_...,227000.00,full_time,38.664532,2026-09-17T07:29:42Z,"{'label': 'IT Jobs', 'tag': 'it-jobs', '__CLAS...",-121.145000,Software Engineer,...,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTg4NzYxNzEwNiIsI...,0,{'__CLASS__': 'Adzuna::API::Response::Company'...,Adzuna::API::Response::Job,IT Jobs,it-jobs,Voyager Technologies,"Folsom, Sacramento County",US,California
4,5885956386,permanent,https://www.adzuna.com/details/5885956386?utm_...,121100.00,full_time,34.508431,2026-09-16T06:30:22Z,"{'label': 'Engineering Jobs', 'tag': 'engineer...",-119.731021,Software Engineer,...,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiM09WQkNkdko4UkdIV...,0,"{'display_name': 'Teledyne Technologies', '__C...",Adzuna::API::Response::Job,Engineering Jobs,engineering-jobs,Teledyne Technologies,"Goleta, Santa Barbara County",US,California


Bước 3:  Data Type Standardization (Chuẩn hóa kiểu dữ liệu)

Sau khi tách các cột có cấu trúc lồng nhau, bước tiếp theo là chuẩn hóa kiểu dữ liệu nhằm đảm bảo các giá trị được lưu trữ đúng định dạng, thống nhất và thuận tiện cho việc xử lý ở các bước tiếp theo.

Các thao tác thực hiện:

Text: Chuyển các cột văn bản sang kiểu string và loại bỏ khoảng trắng thừa ở đầu/cuối bằng str.strip().

Numeric: Chuyển salary_min, salary_max, latitude và longitude sang kiểu dữ liệu số.

Datetime: Chuyển created sang kiểu datetime, sử dụng UTC để thống nhất múi giờ.

Integer: Chuyển salary_is_predicted sang kiểu số nguyên có hỗ trợ giá trị thiếu (Int64).

Kiểm tra lại kiểu dữ liệu sau khi chuyển đổi bằng df.info().

Lưu ý: Sử dụng errors="coerce" để chuyển những giá trị không thể chuyển đổi thành giá trị thiếu (NaN/NaT), thay vì làm gián đoạn quá trình xử lý. Các giá trị này sẽ được kiểm tra ở bước xử lý missing values.

Kết quả mong đợi: Các cột có kiểu dữ liệu phù hợp và nhất quán, tạo nền tảng cho việc kiểm tra dữ liệu trùng lặp, giá trị bất thường và dữ liệu thiếu trong các bước tiếp theo.

In [3]:
# Chuẩn hóa các cột văn bản
text_cols = [
    "title",
    "description",
    "category_label",
    "category_tag",
    "company_name",
    "location_name",
    "location_country",
    "location_state",
    "contract_type",
    "contract_time"
]

for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

df["id"] = df["id"].str.strip()

# Chuẩn hóa numeric
numeric_cols = [
    "salary_min",
    "salary_max",
    "latitude",
    "longitude"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# Chuẩn hóa thời gian
df["created"] = pd.to_datetime(
    df["created"],
    utc=True,
    errors="coerce"
)

# Chuẩn hóa cột trạng thái lương
df["salary_is_predicted"] = pd.to_numeric(
    df["salary_is_predicted"],
    errors="coerce"
).astype("Int64")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2403 entries, 0 to 2402
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   id                   2403 non-null   string             
 1   contract_type        2403 non-null   string             
 2   redirect_url         2403 non-null   str                
 3   salary_max           2403 non-null   float64            
 4   contract_time        2403 non-null   string             
 5   latitude             2131 non-null   float64            
 6   created              2403 non-null   datetime64[us, UTC]
 7   category             2403 non-null   str                
 8   longitude            2131 non-null   float64            
 9   title                2403 non-null   string             
 10  location             2403 non-null   str                
 11  description          2403 non-null   string             
 12  salary_min           2403 non-n

Bước 4:  Duplicate Handling (Kiểm tra và xử lý dữ liệu trùng lặp)

Sau khi chuẩn hóa cấu trúc và kiểu dữ liệu, bước tiếp theo là kiểm tra các bản ghi trùng lặp trong dataset nhằm đảm bảo mỗi tin tuyển dụng được đại diện một cách chính xác, tránh gây sai lệch trong quá trình phân tích và huấn luyện mô hình.

Các thao tác thực hiện:

Kiểm tra số lượng dòng trùng lặp hoàn toàn trong dataset.

Kiểm tra các bản ghi có cùng id (mã định danh tin tuyển dụng).

So sánh nội dung của các bản ghi trùng ID để xác định có sự khác biệt về thông tin công việc, mức lương hoặc các thuộc tính quan trọng hay không.

Chỉ loại bỏ các bản ghi trùng ID khi xác nhận chúng không chứa thông tin khác biệt cần giữ lại.

Kiểm tra lại kích thước dataset và số lượng ID duy nhất sau khi xử lý.

Lưu ý: Không tự động xóa tất cả các bản ghi trùng ID khi chưa kiểm tra nội dung, vì có thể tồn tại những bản ghi được cập nhật thông tin trong quá trình thu thập dữ liệu.

Kết quả mong đợi: Dataset không còn các bản ghi trùng lặp không cần thiết, đồng thời bảo toàn thông tin của từng tin tuyển dụng để phục vụ các bước preprocessing tiếp theo.

In [4]:
# Các cột không dùng để so sánh nội dung tin đăng
exclude_cols = [
    "id",
    "redirect_url",
    "adref",
    "category",
    "company",
    "location",
    "__CLASS__"
]

core_cols = [
    col for col in df.columns
    if col not in exclude_cols
]

# Kiểm tra các ID trùng có nội dung khác nhau không
conflicts = (
    df.groupby("id", dropna=False)[core_cols]
      .nunique(dropna=False)
      .gt(1)
      .any(axis=1)
)

print("Conflicting IDs:", conflicts.sum())

# Chỉ xóa bản ghi trùng khi đã xác nhận không có xung đột
if conflicts.any():
    raise ValueError(
        "Có ID trùng nhưng nội dung khác nhau"
    )

df = df.drop_duplicates(
    subset=["id"],
    keep="first"
).copy()

print("After removing duplicates:", df.shape)
df_clean = df.copy()

Conflicting IDs: 0
After removing duplicates: (1957, 23)


Bước 5:  Salary Validation & Target Creation (Kiểm tra dữ liệu lương và tạo biến mục tiêu)

Sau khi xử lý các bản ghi trùng lặp, bước tiếp theo là kiểm tra tính hợp lệ và nhất quán của dữ liệu lương, đồng thời xây dựng biến mục tiêu salary_mid cho bài toán dự đoán lương.

Các thao tác thực hiện:

Kiểm tra giá trị thiếu (NaN) trong hai cột salary_min và salary_max.

Phát hiện các bản ghi có mức lương không hợp lệ, chẳng hạn lương âm, salary_max <= 0 hoặc salary_min > salary_max.

Tạo cột salary_invalid_flag để đánh dấu những bản ghi có khoảng lương không hợp lệ.

Tạo cột salary_review_flag để đánh dấu các mức lương bất thường cần kiểm tra thêm, đặc biệt là những giá trị quá nhỏ hoặc có khả năng khác đơn vị tính.

Kiểm tra sự khác biệt giữa các mức lương được Adzuna ước tính và không được đánh dấu là ước tính thông qua salary_is_predicted.

Tạo biến mục tiêu salary_mid bằng cách tính trung điểm của khoảng lương hợp lệ:


salary_mid = (salary_min + salary_max) / 2


Lưu ý:

Không tự động loại bỏ mọi giá trị lương bất thường khi chưa xác minh nguyên nhân.

Những bản ghi có khoảng lương không hợp lệ sẽ không được tạo giá trị target hợp lệ.

Cần kiểm tra và thống nhất đơn vị lương (giờ/tháng/năm) trước khi sử dụng dữ liệu để huấn luyện.

salary_min và salary_max chỉ được sử dụng để xây dựng target, không đưa vào features (X) khi huấn luyện mô hình dự đoán salary_mid, nhằm tránh Data Leakage.

Kết quả mong đợi: Dataset có thêm biến mục tiêu salary_mid và các cột đánh dấu chất lượng dữ liệu lương, giúp xác định những bản ghi đủ điều kiện sử dụng trong quá trình huấn luyện mô hình.

In [5]:
# 1. Kiểm tra khoảng lương không hợp lệ
df_clean["salary_invalid_flag"] = (
    df_clean["salary_min"].isna()
    | df_clean["salary_max"].isna()
    | (df_clean["salary_min"] < 0)
    | (df_clean["salary_max"] <= 0)
    | (df_clean["salary_min"] > df_clean["salary_max"])
)
# Ngưỡng rà soát được lựa chọn từ phân phối dữ liệu
SALARY_REVIEW_THRESHOLD = 10000

# 2. Đánh dấu những bản ghi cần rà soát
df_clean["salary_review_flag"] = (
    df_clean["salary_invalid_flag"]
    | (df_clean["salary_min"] <= 0)
    | (df_clean["salary_max"] < SALARY_REVIEW_THRESHOLD)
)

# 3. Tạo target salary_mid
df_clean["salary_mid"] = (
    (df_clean["salary_min"] + df_clean["salary_max"]) / 2
).where(~df_clean["salary_invalid_flag"]).round(2)

# 4. Kiểm tra
print("Invalid salary:", df_clean["salary_invalid_flag"].sum())
print("Need review:", df_clean["salary_review_flag"].sum())

display(
    df_clean[
        ["salary_min", "salary_max", "salary_mid"]
    ].head()
)

Invalid salary: 0
Need review: 12


,salary_min,salary_max,salary_mid
0,312845.06,312845.06,312845.06
1,59500.00,99000.00,79250.00
2,106080.02,106080.02,106080.02
3,104000.00,227000.00,165500.00
4,90800.00,121100.00,105950.00


Note: có 12 dữ liệu cần kiểm tra 

Các bản ghi có salary_max < 10000 được đánh dấu để kiểm tra do có khả năng sử dụng đơn vị lương khác (theo giờ, tuần hoặc tháng) hoặc chứa giá trị bất thường.

Kết quả: Có 12/1.957 bản ghi (khoảng 0,61%) thuộc nhóm cần rà soát.

Do mục tiêu nghiên cứu là xây dựng mô hình baseline dự đoán trung điểm lương năm (salary_mid), nhóm bản ghi này được tạm loại khỏi tập dữ liệu huấn luyện và lưu riêng để phục vụ việc xác minh trong tương lai.

Lưu ý: Ngưỡng $10,000 được xác định dựa trên đặc điểm phân phối của dataset, không phải mức lương tối thiểu theo quy định tại Mỹ. Việc loại bỏ nhóm giá trị thấp chỉ nhằm giảm nguy cơ không nhất quán về đơn vị lương, không chứng minh rằng toàn bộ dữ liệu còn lại đều đã được xác nhận là lương năm.

In [6]:
# Lưu riêng các bản ghi cần review
df_review = df_clean.loc[
    df_clean["salary_review_flag"]
].copy()

# Loại bỏ các bản ghi invalid hoặc cần review
df_clean = df_clean.loc[
    ~df_clean["salary_invalid_flag"]
    & ~df_clean["salary_review_flag"]
].copy()

# Kiểm tra kết quả
print("Excluded records:", len(df_review))
print("Remaining records:", len(df_clean))

Excluded records: 12
Remaining records: 1945


Bước 6:  Missing Values Handling (Kiểm tra và xử lý dữ liệu thiếu)

Sau khi hoàn thành bước kiểm tra dữ liệu lương, tạo biến mục tiêu salary_mid và loại bỏ các bản ghi không phù hợp, bước tiếp theo là kiểm tra tình trạng dữ liệu thiếu (Missing Values) trong dataset.

Bước này nhằm xác định những thuộc tính chưa có đầy đủ thông tin và lựa chọn phương pháp xử lý phù hợp, đồng thời hạn chế việc làm mất dữ liệu không cần thiết.

Các thao tác thực hiện:

Thống kê số lượng (Missing Count) và tỷ lệ (Missing Percent) giá trị thiếu của từng cột.

Xác định các thuộc tính có missing values và đánh giá mức độ ảnh hưởng đến bài toán dự đoán lương.

Đặc biệt kiểm tra hai thuộc tính địa lý latitude và longitude, do một số tin tuyển dụng không cung cấp tọa độ cụ thể.

Kiểm tra lại biến mục tiêu salary_mid để đảm bảo không còn giá trị thiếu trước khi huấn luyện.

Giữ nguyên những giá trị thiếu chưa cần xử lý trực tiếp, thay vì tự động thay thế bằng 0 hoặc loại bỏ toàn bộ bản ghi.
Kết quả mong đợi: Xác định đầy đủ tình trạng missing values, đảm bảo biến mục tiêu salary_mid không bị thiếu và xây dựng được chiến lược xử lý phù hợp cho từng nhóm thuộc tính trước khi huấn luyện mô hình.

In [7]:
# 1. Thống kê số lượng và tỷ lệ Missing Values
missing_report = pd.DataFrame({
    "Missing Count": df_clean.isna().sum(),
    "Missing Percent": (df_clean.isna().mean() * 100).round(2)
}).sort_values(
    by="Missing Count",
    ascending=False
)

print("Current dataset shape:", df_clean.shape)
display(missing_report)


# 2. Hiển thị các cột thực sự có Missing Values
print("Columns with missing values:")

display(
    missing_report.loc[
        missing_report["Missing Count"] > 0
    ]
)


# 3. Kiểm tra Missing Values của Target
missing_target = df_clean["salary_mid"].isna().sum()

print("Missing salary_mid:", missing_target)

assert missing_target == 0, "Target salary_mid contains missing values!"


# 4. Kiểm tra các bản ghi thiếu location_state
print("Records with missing location_state:")

display(
    df_clean.loc[
        df_clean["location_state"].isna(),
        [
            "id",
            "title",
            "location_name",
            "location_country",
            "location_state"
        ]
    ]
)


# 5. Kiểm tra lại số lượng dữ liệu
print("Final dataset shape:", df_clean.shape)


Current dataset shape: (1945, 26)


,Missing Count,Missing Percent
longitude,219,11.26
latitude,219,11.26
location_state,1,0.05
contract_type,0,0.00
salary_max,0,0.00
id,0,0.00
contract_time,0,0.00
created,0,0.00
category,0,0.00
redirect_url,0,0.00


Columns with missing values:


,Missing Count,Missing Percent
longitude,219,11.26
latitude,219,11.26
location_state,1,0.05


Missing salary_mid: 0
Records with missing location_state:


,id,title,location_name,location_country,location_state
2157,5899707926,"Channel Account Manager, AT&T",US,US,<NA>


Final dataset shape: (1945, 26)


Bước 6:  Missing Values & Salary Source Assessment (Kiểm tra dữ liệu thiếu và nguồn gốc mức lương)

Sau bước Salary Validation, dataset còn 1.945 bản ghi với biến mục tiêu salary_mid. Bước này nhằm kiểm tra tình trạng dữ liệu thiếu và phân tích nguồn gốc của các giá trị lương để chuẩn bị cho quá trình huấn luyện mô hình.

Các thao tác thực hiện:

Thống kê số lượng (Missing Count) và tỷ lệ (Missing Percent) giá trị thiếu của từng thuộc tính.

Xác định các cột có missing values và kiểm tra riêng biến mục tiêu salary_mid.

Kiểm tra các bản ghi thiếu thông tin location_state.

Phân tích phân bố của salary_is_predicted để phân biệt hai nhóm dữ liệu:

0: Mức lương không được Adzuna đánh dấu là ước tính.

1: Mức lương được Adzuna ước tính.

Giữ lại cả hai nhóm nhằm duy trì số lượng dữ liệu cho quá trình huấn luyện.

Chiến lược xử lý:

latitude và longitude sẽ được loại bỏ ở Cell 7 do mô hình baseline không sử dụng tọa độ địa lý.

Các giá trị thiếu còn lại được giữ nguyên để xử lý trong ML Pipeline nếu cần.

Không thực hiện imputation trên toàn bộ dataset nhằm tránh Data Leakage.

Giữ salary_is_predicted để áp dụng kỹ thuật Sample Weighting trong notebook 05_model_training.ipynb.

Đối với Sample Weighting, nghiên cứu dự kiến ưu tiên các bản ghi không được Adzuna đánh dấu là ước tính bằng cách gán trọng số huấn luyện cao hơn. Giá trị trọng số cụ thể sẽ được thử nghiệm và lựa chọn dựa trên kết quả validation.

Thuộc tính salary_is_predicted chỉ phục vụ việc xác định trọng số và đánh giá chất lượng nhãn, không được sử dụng làm feature đầu vào (X) của mô hình.

Kết quả mong đợi: Xác định tình trạng missing values, đảm bảo target salary_mid không bị thiếu và giữ nguyên cả hai nhóm dữ liệu lương để phục vụ Sample Weighting ở giai đoạn Training.

In [8]:
# 1. Thống kê số lượng và tỷ lệ Missing Values
missing_report = pd.DataFrame({
    "Missing Count": df_clean.isna().sum(),
    "Missing Percent": (df_clean.isna().mean() * 100).round(2)
}).sort_values(
    by="Missing Count",
    ascending=False
)

print("Current dataset shape:", df_clean.shape)
display(missing_report)


# 2. Hiển thị các cột thực sự có Missing Values
print("Columns with missing values:")

display(
    missing_report.loc[
        missing_report["Missing Count"] > 0
    ]
)


# 3. Kiểm tra Missing Values của Target
missing_target = df_clean["salary_mid"].isna().sum()

print("Missing salary_mid:", missing_target)

assert missing_target == 0, (
    "Target salary_mid contains missing values!"
)


# 4. Kiểm tra bản ghi thiếu location_state
print("Records with missing location_state:")

display(
    df_clean.loc[
        df_clean["location_state"].isna(),
        [
            "id",
            "title",
            "location_name",
            "location_country",
            "location_state"
        ]
    ]
)


# 5. Kiểm tra nguồn gốc dữ liệu lương
assert df_clean["salary_is_predicted"].notna().all()

assert df_clean["salary_is_predicted"].isin([0, 1]).all()

source_summary = (
    df_clean["salary_is_predicted"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("salary_is_predicted")
    .reset_index(name="Count")
)

source_summary["Percent"] = (
    source_summary["Count"] / len(df_clean) * 100
).round(2)

print("Salary Source Distribution:")
display(source_summary)


# 6. Kiểm tra dữ liệu sau cùng
print("Final dataset shape:", df_clean.shape)

Current dataset shape: (1945, 26)


,Missing Count,Missing Percent
longitude,219,11.26
latitude,219,11.26
location_state,1,0.05
contract_type,0,0.00
salary_max,0,0.00
id,0,0.00
contract_time,0,0.00
created,0,0.00
category,0,0.00
redirect_url,0,0.00


Columns with missing values:


,Missing Count,Missing Percent
longitude,219,11.26
latitude,219,11.26
location_state,1,0.05


Missing salary_mid: 0
Records with missing location_state:


,id,title,location_name,location_country,location_state
2157,5899707926,"Channel Account Manager, AT&T",US,US,<NA>


Salary Source Distribution:


,salary_is_predicted,Count,Percent
0,0,946,48.64
1,1,999,51.36


Final dataset shape: (1945, 26)


In [9]:
print(df.head())

           id contract_type  \
0  5904487469     permanent   
1  5903572359     permanent   
2  5879639363     permanent   
3  5887617106     permanent   
4  5885956386     permanent   

                                        redirect_url  salary_max  \
0  https://www.adzuna.com/land/ad/5904487469?se=3...   312845.06   
1  https://www.adzuna.com/details/5903572359?utm_...    99000.00   
2  https://www.adzuna.com/details/5879639363?utm_...   106080.02   
3  https://www.adzuna.com/details/5887617106?utm_...   227000.00   
4  https://www.adzuna.com/details/5885956386?utm_...   121100.00   

  contract_time   latitude                   created  \
0     full_time  37.393080 2026-09-30 22:39:48+00:00   
1     full_time  33.751486 2026-09-30 02:32:52+00:00   
2     full_time  37.798085 2026-09-11 07:15:48+00:00   
3     full_time  38.664532 2026-09-17 07:29:42+00:00   
4     full_time  34.508431 2026-09-16 06:30:22+00:00   

                                            category   longitude  \

7. Drop Unnecessary Columns (Loại bỏ các thuộc tính không cần thiết)

Sau khi hoàn thành các bước kiểm tra và làm sạch dữ liệu, nghiên cứu tiến hành loại bỏ những thuộc tính dư thừa nhằm tinh giản dataset và chuẩn bị dữ liệu cho giai đoạn Feature Engineering và Model Training.

Các thao tác thực hiện:

Loại bỏ các cột metadata như redirect_url, adref và __CLASS__ do không mang thông tin cần thiết cho bài toán dự đoán.

Loại bỏ các cột dictionary gốc (category, company, location) và các cột trung gian sau khi đã trích xuất thông tin cần thiết.

Chỉ giữ lại location_state để biểu diễn thông tin địa lý, do nghiên cứu tập trung dự đoán mức lương theo từng bang tại Mỹ (State-Level Salary Prediction). Các thuộc tính latitude, longitude, location_name và location_country sẽ được loại bỏ.

Loại bỏ category_tag do đã giữ lại thông tin ngành nghề trong category_label.

Loại bỏ salary_min và salary_max sau khi đã xây dựng biến mục tiêu salary_mid, nhằm tránh nguy cơ Data Leakage trong quá trình huấn luyện.

Loại bỏ các cột đánh dấu dữ liệu bất thường (salary_invalid_flag, salary_review_flag) sau khi hoàn thành Salary Validation.

Các thuộc tính đặc biệt được giữ lại:

salary_mid: Biến mục tiêu (Target) của mô hình dự đoán lương năm.

salary_is_predicted: Phục vụ kỹ thuật Sample Weighting trong giai đoạn Model Training, không sử dụng làm feature đầu vào.

id: Phục vụ truy xuất và đối chiếu dữ liệu gốc, không sử dụng làm feature khi huấn luyện.

location_state: Thuộc tính địa lý chính, đại diện cho bang nơi đăng tuyển dụng.

Lưu ý: Bước này chỉ thực hiện lựa chọn và loại bỏ thuộc tính, chưa tiến hành Encoding, Scaling hoặc tách Features (X) và Target (y). Các thao tác này sẽ được triển khai trong ML Pipeline ở giai đoạn Model Training.

Kết quả mong đợi: Dataset còn lại 1.945 bản ghi với 11 thuộc tính, được tinh giản và bảo toàn các thông tin cần thiết cho bài toán dự đoán trung điểm lương năm theo bang tại Mỹ.

In [10]:
# 1. Danh sách các cột cần loại bỏ
drop_cols = [
    # Metadata
    "redirect_url",
    "adref",
    "__CLASS__",

    # Các cột dictionary gốc và trung gian
    "category",
    "company",
    "location",
    "category_obj",
    "company_obj",
    "location_obj",

    # Chỉ giữ location_state để dự đoán theo bang
    "latitude",
    "longitude",
    "location_name",
    "location_country",

    # Các thuộc tính dư thừa
    "category_tag",

    # Các cột lương đã dùng để tạo target
    "salary_min",
    "salary_max",

    # Các cột kiểm tra chất lượng dữ liệu
    "salary_invalid_flag",
    "salary_review_flag",
    "review_reason"
]

# 2. Loại bỏ các cột không cần thiết
df_clean = df_clean.drop(
    columns=drop_cols,
    errors="ignore"
).copy()

# 3. Reset Index
df_clean = df_clean.reset_index(drop=True)

# 4. Kiểm tra kết quả
print("Final Dataset Shape:", df_clean.shape)
print("Remaining Columns:", df_clean.columns.tolist())

# 5. Hiển thị DataFrame sau Preprocessing
display(df_clean.head(10))


Final Dataset Shape: (1945, 11)
Remaining Columns: ['id', 'contract_type', 'contract_time', 'created', 'title', 'description', 'salary_is_predicted', 'category_label', 'company_name', 'location_state', 'salary_mid']


,id,contract_type,contract_time,created,title,description,salary_is_predicted,category_label,company_name,location_state,salary_mid
0,5904487469,permanent,full_time,2026-09-30 22:39:48+00:00,Principal Software Engineer,Microsoft and OpenAI have formed one of the mo...,1,IT Jobs,Microsoft,California,312845.06
1,5903572359,permanent,full_time,2026-09-30 02:32:52+00:00,Software Engineer,ETAP empowers customers to make informed decis...,0,Engineering Jobs,ETAP,California,79250.00
2,5879639363,permanent,full_time,2026-09-11 07:15:48+00:00,Software Engineer,"Responsibilities Design, implement, and mainta...",1,IT Jobs,Pivot Robotics,California,106080.02
3,5887617106,permanent,full_time,2026-09-17 07:29:42+00:00,Software Engineer,"Voyager is an innovative space, defense, and n...",0,IT Jobs,Voyager Technologies,California,165500.00
4,5885956386,permanent,full_time,2026-09-16 06:30:22+00:00,Software Engineer,Be visionary Teledyne Technologies Incorporate...,0,Engineering Jobs,Teledyne Technologies,California,105950.00
5,5750278668,permanent,full_time,2026-06-03 03:54:07+00:00,Software Engineer,About Cowboy Space Corp: Mission Cowboy Space ...,0,Engineering Jobs,Cowboy Space,California,150000.00
6,5918771493,permanent,full_time,2026-10-08 06:53:37+00:00,Software Engineer,Worker Type Regular Job Description Summary As...,0,IT Jobs,AeroVironment,California,103500.00
7,5930009790,permanent,full_time,2026-10-16 00:39:22+00:00,Director Of Software Engineering,Job Title: Director of Software Engineering Jo...,1,IT Jobs,Actalent,California,207465.56
8,5918356032,permanent,full_time,2026-10-07 23:29:49+00:00,Senior Software Engineer,Job Title: Senior Software Engineer Job Descri...,1,Engineering Jobs,Actalent,California,140106.31
9,5919718766,permanent,full_time,2026-10-08 19:38:10+00:00,Embedded Software Engineer,Embedded Software Engineer- hybrid schedule! J...,1,Engineering Jobs,Actalent,California,119511.04


Bước 8:  Final Validation & Save Cleaned Dataset

Sau khi hoàn thành quá trình Preprocessing, nghiên cứu thực hiện kiểm tra lần cuối nhằm đảm bảo dataset đáp ứng các điều kiện cần thiết trước khi chuyển sang giai đoạn Feature Engineering.

Các nội dung kiểm tra bao gồm:

Kiểm tra kích thước và cấu trúc dataset sau khi loại bỏ các thuộc tính dư thừa.

Đảm bảo không còn bản ghi trùng lặp theo id.

Đảm bảo biến mục tiêu salary_mid không chứa giá trị thiếu.

Kiểm tra tính hợp lệ của salary_is_predicted để phục vụ Sample Weighting.

Xác nhận các thuộc tính quan trọng được giữ lại đầy đủ.

Sau khi kiểm tra thành công, dataset được lưu dưới dạng CSV tại thư mục data/processed/, làm dữ liệu đầu vào cho các notebook tiếp theo.

In [11]:
from pathlib import Path

# 1. Final Validation
assert df_clean.shape == (1945, 11), "Unexpected dataset shape!"
assert df_clean["id"].is_unique, "Duplicate IDs found!"
assert df_clean["salary_mid"].notna().all(), "Missing target found!"
assert df_clean["salary_is_predicted"].notna().all()
assert df_clean["salary_is_predicted"].isin([0, 1]).all()

# 2. Kiểm tra các cột quan trọng
required_cols = [
    "id",
    "title",
    "description",
    "location_state",
    "salary_is_predicted",
    "salary_mid"
]

assert set(required_cols).issubset(df_clean.columns)

print("Final validation passed!")
print("Final Dataset Shape:", df_clean.shape)

# 3. Lưu Cleaned Dataset
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "cleaned_adzuna.csv"

df_clean.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Dataset saved to:", output_path)


Final validation passed!
Final Dataset Shape: (1945, 11)
Dataset saved to: ..\data\processed\cleaned_adzuna.csv
